# Fine-tune and Deploy YOLO26-OBB on AMD Radeon Cloud

In this hands-on workshop, we use **Ultralytics YOLO26** on one **AMD Radeon PRO W7900D** to build an end-to-end oriented object detection workflow:

**COCO HBB weights → Sheep OBB fine-tuning → separate val/test evaluation → ONNX export → MIGraphX acceleration → real-time OBB tracking**

By the end, you will know how the same Ultralytics workflow connects training and deployment on AMD Radeon Cloud.

## Hands-on roadmap

1. Check the Ultralytics and AMD GPU environment.
2. Inspect the prepared AUTH-Sheep and SheepCounter OBB data.
3. Transfer a COCO HBB checkpoint into a one-class OBB model and fine-tune it on one GPU.
4. Validate on SheepCounter and held-out videos, then visualize the downstream adaptation.
5. Export to ONNX, verify MIGraphX, and benchmark steady-state inference.
6. Deploy YOLO26n-OBB + ByteTrack as a real-time video stream.

The prepared dataset, fallback checkpoint, and container keep the workflow reproducible and leave time for questions and exploration.

## 1. Check the environment

Confirm that Ultralytics loads correctly and the AMD GPU is available.

In [ ]:
import ultralytics

ultralytics.checks()

!rocm-smi --showuse --showmeminfo vram


## 2. Build the Sheep OBB dataset

We combine two public aerial datasets and keep the split easy to understand:

| Dataset | Original data | Workshop split |
|---|---|---|
| [SheepCounter v11](https://universe.roboflow.com/riisprivate/sheepcounter/dataset/11) | 1,220 train + 507 valid images | 1,220 train; 507 validation |
| [AUTH-Sheep](https://github.com/idmt-odoll/AUTH-Sheep) | 4 annotated UAV videos | `video1` + `video4`: 198 train; `video2` + `video3`: 154 test frames |

All four AUTH-Sheep videos are sampled every 15 frames. The result is **1,418 training images**, **507 validation images**, and **154 test frames**. For this workshop, we keep only the `sheep` annotations.


### Explore the training data

Use the slider below each preview or type an image number directly into its input box. AUTH-Sheep and SheepCounter update independently, and recently viewed samples stay cached.


In [ ]:
from utils.workshop_utils import browse_obb_train_samples

browse_obb_train_samples("/datasets/sheep-datasets")


### Inspect the prepared dataset

The next cell shows the training-ready image and label folders with their file counts, then displays the YAML that connects those splits to the single `sheep` class. Labels use Ultralytics OBB format: `class x1 y1 x2 y2 x3 y3 x4 y4`, with four normalized corner points.


In [ ]:
from pathlib import Path
from utils.workshop_utils import show_dataset_tree

print("Dataset directory tree:\n")
show_dataset_tree("/datasets/sheep-datasets")

print("\n" + "─" * 64 + "\n")
print("Dataset YAML: config/sheep.yaml\n")
print(Path("config/sheep.yaml").read_text())


## 3. Transfer COCO HBB features to Sheep OBB

The source checkpoint is `yolo26n.pt`: an 80-class COCO horizontal-box detector. The target architecture is `yolo26n-obb.yaml`: a one-class oriented detector. Ultralytics transfers all shape-compatible features; the task-specific classification and angle components adapt during fine-tuning.

This deliberately demonstrates a real downstream change: **general HBB detection → aerial single-class OBB detection**.

### Preview the pretrained COCO model

Use the controls below to browse `sheep` predictions from the pretrained COCO HBB model on familiar COCO images and the new AUTH-Sheep aerial domain. Each title reports the number of detected sheep, and predictions are cached after their first run.


In [ ]:
from ultralytics import YOLO
from utils.workshop_utils import browse_pretrained_coco_predictions

browse_pretrained_coco_predictions(YOLO("/models/yolo26n.pt"))


### Initialize the OBB model

The next cell creates a one-class YOLO26n-OBB model and loads all shape-compatible features from the pretrained COCO checkpoint.


In [ ]:
model = YOLO("yolo26n-obb.yaml").load("/models/yolo26n.pt")


## 4. Fine-tune on one W7900D

The three workshop knobs stay visible in `model.train()`: training duration, input size, and batch size. Advanced optimizer and runtime defaults live in `train_defaults.yaml` and are expanded with `**train_defaults`.

On one W7900D, the reference 12-epoch run completes in **about 300 seconds**. Set `run_training = False` only when live training cannot start; the notebook will then load the prepared COCO-initialized trained checkpoint.


In [ ]:
import time
from ultralytics.utils import YAML

run_training = True

if run_training:
    started = time.perf_counter()
    results = model.train(
        data="config/sheep.yaml",
        epochs=12,
        imgsz=832,
        batch=64,
        **YAML.load("config/train_defaults.yaml"),
    )
    best_weights = Path(results.save_dir) / "weights/best.pt"
    print(f"Training wall time: {time.perf_counter() - started:.1f} seconds")
else:
    best_weights = Path("/models/yolo26n_obb_sheep_best.pt")

print("\n" + "─" * 72)
tuned = YOLO(best_weights)
precision = str(next(tuned.model.parameters()).dtype).replace("torch.bfloat", "BF").replace("torch.float", "FP")
print("Fine-tuned model loaded successfully")
print(f"  Weights:    {best_weights}")
print(f"  Task:       {tuned.task.upper()}")
print(f"  Precision:  {precision}")
print(f"  Parameters: {sum(parameter.numel() for parameter in tuned.model.parameters()):,}")

## 5. Evaluate the fine-tuned Sheep OBB model

### Quantitative evaluation

The `val` split contains 507 SheepCounter images. The `test` split contains 154 frames sampled every 15 frames from AUTH-Sheep `video2` and `video3`; neither video contributes any frame to training. We evaluate the two sources separately so their results remain easy to interpret.


In [ ]:
import pandas as pd

sheepcounter = tuned.val(data="config/sheep.yaml", split="val", imgsz=832, batch=64, end2end=False).box
auth_sheep = tuned.val(data="config/sheep.yaml", split="test", imgsz=832, batch=64, end2end=False).box

pd.DataFrame([
    ["val", "SheepCounter", 507, sheepcounter.mp, sheepcounter.mr, sheepcounter.map50, sheepcounter.map],
    ["test", "AUTH-Sheep video2 + video3", 154, auth_sheep.mp, auth_sheep.mr, auth_sheep.map50, auth_sheep.map],
], columns=["Split", "Dataset", "Images", "Precision", "Recall", "mAP50", "mAP50-95"]).round(3)


### Qualitative before and after fine-tuning

Use the linked controls to browse the same held-out aerial frame through both models. COCO already contains the semantic class *sheep*, but its checkpoint was trained for natural-image HBB detection; the side-by-side results make the domain and task adaptation visible. The fine-tuned preview uses the same 640×640 FP16 standard OBB decoding/NMS path as the later MIGraphX example. This is a qualitative comparison—not a shared mAP evaluation—because HBB and OBB are different output tasks.


In [ ]:
from utils.workshop_utils import browse_finetuning_comparison

browse_finetuning_comparison(YOLO("/models/yolo26n.pt"), YOLO(best_weights))


## 6. Accelerate Sheep OBB inference with ONNX and MIGraphX

This section exports the fine-tuned model, runs it with AMD MIGraphX, checks that acceleration preserves its predictions, and measures the resulting performance gain.

### Export the fine-tuned OBB model to ONNX

Export the raw OBB network at a deployment resolution of 640×640. Ultralytics keeps rotated-box decoding and NMS outside the ONNX graph, avoiding task-specific Top-K and rotated post-processing operators in the graph.

In [ ]:
ONNX_MODEL = Path(YOLO(best_weights).export(
    format="onnx",
    imgsz=640,
    simplify=False,
    end2end=False,
))
print("Exported model:", ONNX_MODEL)

### Run accelerated ONNX inference with MIGraphX

Load the exported network with `MIGraphXExecutionProvider` in FP16 mode. Ultralytics still handles image preprocessing, OBB decoding, and rotated NMS. After the one-time graph compilation, the cell prints a compact inference summary and displays the resulting oriented boxes.

In [ ]:
import onnxruntime as ort

SAMPLE_IMAGE = Path("/datasets/sheep-datasets/images/test") / "auth_v3_00330.jpg"
migraphx = YOLO(ONNX_MODEL, task="obb")
migraphx_result = migraphx.predict(
    SAMPLE_IMAGE, imgsz=640, quantize="fp16", end2end=False
)[0]
backend = migraphx.predictor.model.backend

print("\nMIGraphX inference verified")
print("─" * 64)
print(f"Execution path  Ultralytics → ONNX Runtime {ort.__version__} → {backend.provider} → Radeon GPU")
print(f"Model           {ONNX_MODEL.name}")
print(f"Precision       {'FP16' if backend.migraphx_fp16 else 'FP32'}")
print(f"Network input   {' × '.join(map(str, backend.session.get_inputs()[0].shape))}")
print(f"Prediction      {len(migraphx_result.obb)} sheep OBBs · {SAMPLE_IMAGE.name}")
display(migraphx_result.plot(pil=True, labels=False, conf=False, line_width=2))

### Compare PyTorch and MIGraphX accuracy

Evaluate both backends with the same 640×640 FP16 model settings and standard OBB decoding/NMS on SheepCounter `val` and the fully held-out AUTH-Sheep `test` split. The exported ONNX graph has a static batch size of 1, so both backends use batch 1 for a direct numerical comparison.


In [ ]:
pytorch_eval = YOLO(best_weights)
accuracy_rows = []

for split, dataset, images in (("val", "SheepCounter", 507), ("test", "AUTH-Sheep video2 + video3", 154)):
    for backend_name, eval_model in (("PyTorch", pytorch_eval), ("MIGraphX", migraphx)):
        metrics = eval_model.val(
            data="config/sheep.yaml",
            split=split,
            imgsz=640,
            batch=1,
            quantize="fp16",
            end2end=False,
        ).box
        accuracy_rows.append([split, dataset, images, backend_name, metrics.mp, metrics.mr, metrics.map50, metrics.map])

pd.DataFrame(accuracy_rows, columns=["Split", "Dataset", "Images", "Backend", "Precision", "Recall", "mAP50", "mAP50-95"]).round(3)


### Compare inference performance

Measure `YOLO.predict()` throughput with **PyTorch FP32 and FP16 at latency-oriented batch 1 and throughput-oriented batch 32**, plus **MIGraphX FP32 and FP16 at its exported batch 1**. Images per second keeps the batches comparable. Timing excludes file I/O, drawing, model setup, and one-time MIGraphX graph compilation.


In [ ]:
import time
import torch
from utils.acceleration_utils import plot_backend_benchmark

def measure(model, source, quantize, warmup=3, repeats=20):
    def predict():
        model.predict(
            source, imgsz=640, batch=len(source), quantize=quantize, end2end=False,
            verbose=False,
        )

    for _ in range(warmup):
        predict()
    torch.cuda.synchronize()
    started = time.perf_counter()
    for _ in range(repeats):
        predict()
    torch.cuda.synchronize()
    latency = (time.perf_counter() - started) * 1000 / repeats
    return latency, len(source) * 1000 / latency

images = {batch: [migraphx_result.orig_img] * batch for batch in (1, 32)}
pytorch_fp32, pytorch_fp16 = YOLO(best_weights), YOLO(best_weights)
migraphx_fp32 = YOLO(ONNX_MODEL, task="obb")
cases = (
    ("PyTorch", "FP32", pytorch_fp32, images[1], "fp32"),
    ("PyTorch", "FP16", pytorch_fp16, images[1], "fp16"),
    ("PyTorch", "FP32", pytorch_fp32, images[32], "fp32"),
    ("PyTorch", "FP16", pytorch_fp16, images[32], "fp16"),
    ("MIGraphX", "FP32", migraphx_fp32, images[1], "fp32"),
    ("MIGraphX", "FP16", migraphx, images[1], "fp16"),
)

rows = []
for backend_name, precision, model, source, quantize in cases:
    latency, throughput = measure(model, source, quantize)
    rows.append([backend_name, precision, len(source), latency, throughput])

benchmark_results = pd.DataFrame(rows, columns=["Backend", "Precision", "Batch", "Latency (ms/batch)", "Throughput (images/s)"])
plot_backend_benchmark(benchmark_results)


## 7. Deploy a real-time OBB tracking stream

The Notebook opens the video with OpenCV and explicitly calls Ultralytics `model.track()` for every frame. `LiveCanvas` handles only OBB rendering, source-FPS pacing, and bandwidth-aware delivery to a Jupyter Image widget.

Change `VIDEO_NAME` to switch between the two fully held-out videos. The selected video runs from beginning to end.

In [ ]:
import cv2
from utils.live_tracking_demo import LiveCanvas

VIDEO_NAME = "video2"  # Choose "video2" or "video3"
video = cv2.VideoCapture(f"/datasets/sheep-datasets/raw/AUTH-Sheep/{VIDEO_NAME}.mp4")
source_fps = video.get(cv2.CAP_PROP_FPS) or 30.0
track_model = YOLO(best_weights)
canvas = LiveCanvas(fps=source_fps, size=(960, 540))

while video.isOpened():
    success, frame = video.read()
    if not success:
        break

    frame = cv2.resize(frame, canvas.size)
    tracked = track_model.track(
        frame,
        persist=True,
        tracker="config/bytetrack_sheep_workshop.yaml",
        imgsz=832,
        quantize="fp16",
        end2end=False,
        verbose=False,
    )[0]
    canvas.write(tracked)

video.release()
canvas.finish()


## Wrap-up and free exploration

You have completed one continuous Ultralytics workflow on AMD Radeon Cloud:

- initialized an OBB task from general COCO HBB features;
- fine-tuned and validated on exactly one W7900D;
- exported the model to ONNX and verified MIGraphX execution;
- measured steady-state acceleration;
- deployed real-time oriented detection and ByteTrack tracking at the source frame rate.

Try switching `VIDEO_NAME`, or changing the confidence, image size, and tracker thresholds. The key pattern remains the same: **Ultralytics connects data, training, validation, export, and deployment through a compact Python API.**